## Lecture 14 - Monsters and mixtures

McElreath's lectures for today: https://www.youtube.com/watch?v=p7g-CgGCS34

McElreath's lectures for the whole book are available here: https://xcelab.net/rm/statistical-rethinking/

An R/Stan repo of code is available here: https://vincentarelbundock.github.io/rethinking2/

An excellent port to Python/PyMC Code is available here: https://github.com/dustinstansbury/statistical-rethinking-2023/tree/main

You are encouraged to work through both of these versions to re-enforce what we're doing in class.

In [ ]:
# Load R packages
library(cmdstanr)    # R interface to Stan
library(posterior)   # Working with posterior draws
library(bayesplot)   # Plotting posterior draws
library(loo)         # WAIC and PSIS-LOO cross-validation

# Helper functions
stdize <- function(x) (x - mean(x))/sd(x)

indexall <- function(L){
    poo <- unique(L)
    Ix <- as.integer(factor(L, levels = poo))
    list(poo, Ix)
}

softmax <- function(x) exp(x - max(x))/sum(exp(x - max(x)))

# Overlay the posterior histograms of each column of a matrix
plot_hists <- function(mat, labels = NULL, xlab = "", vline = NULL,
                       cols = c("dodgerblue", "orange", "forestgreen")){
    mat <- as.matrix(mat)
    # Bin width set by the narrowest column, so each posterior is well resolved
    bw <- min(apply(mat, 2, function(x) diff(range(x))))/20
    brks <- seq(min(mat) - bw, max(mat) + 2*bw, by = bw)
    hs <- lapply(1:ncol(mat), function(i) hist(mat[, i], breaks = brks, plot = FALSE))
    ymax <- max(sapply(hs, function(h) max(h$density)))
    plot(0, 0, type = "n", xlim = range(mat), ylim = c(0, ymax),
         xlab = xlab, ylab = "Density", main = "")
    for (i in 1:ncol(mat)) plot(hs[[i]], freq = FALSE, add = TRUE,
                                col = adjustcolor(cols[i], 0.6), border = NA)
    if (!is.null(vline)) abline(v = vline, lty = 3)
    if (!is.null(labels)) legend("topright", legend = labels, bty = "n",
                                 fill = adjustcolor(cols[1:ncol(mat)], 0.6))
}

# Stan's sampler with the same defaults as pm.sample(1000): 4 chains, 1000 warmup, 1000 draws
sample_stan <- function(model, data){
    model$sample(data = data, chains = 4, parallel_chains = 4, iter_warmup = 1000,
                 iter_sampling = 1000, refresh = 0, show_exceptions = FALSE)
}

# Save the current figure to file (uncomment the savefig() calls below to use)
savefig <- function(file, width = 7, height = 5){
    dev.copy(jpeg, file, width = width, height = height, units = "in", res = 300)
    invisible(dev.off())
}

# Survival Analysis

A key relationship between count models like the Poisson and the data they represent is that the underlying parameters are about rates - how long do we wait (or how wide an area do we search) on average for an event to take place? Therefore another option for quantifying these things lies in modeling times to events - with the trick being that we have to also pay attention to the *censoring* of data. 

1. Left-censored: when you don't know when time started relative to events
2. Right-censored: ending the observation period before the next event occurs
3. Dual-censored: both problems at once

Ignoring the censoring process will also lead to inferrential mistakes, so each survival analysis needs to keep track of their potential effects.

By way of example, let's have a look at some cat data from Austin Texas.


In [ ]:
# Import cat data
cdata <- read.csv('AustinCats.csv')

In [ ]:
head(cdata)

We can see here the data has dates in and out for individual cats, as well as things about their breed etc. Let's say we're interested in the average time to adoption, a useful metric if you're running a cat shelter. If we were modelling just the days to adoption ($D$) and ignoring the censoring, the model would simply be

$$
\begin{align}
D_i \sim & Exponential(\lambda_i)
\end{align}
$$

the underlying rate of adoptions. Let's run that model and see what happens:


In [ ]:
# Adoption days
D <- cdata$days_to_event

In [ ]:
adopt_code <- "
data {
  int<lower=0> N;
  vector[N] days;
}
parameters {
  real<lower=0> Adoption_rate;   // Adoption rate
}
model {
  // Adoption rate
  Adoption_rate ~ exponential(1);

  // Data likelihood
  days ~ exponential(Adoption_rate);
}
"
adopt <- cmdstan_model(write_stan_file(adopt_code))

In [ ]:
trace_a <- sample_stan(adopt, list(N = length(D), days = D))

In [ ]:
mcmc_combo(trace_a$draws("Adoption_rate"), combo = c("dens_overlay", "trace"))
# ggsave('rate.jpg', dpi = 300)

This adoption rate is 0.04 - what does that mean? Well the mean of an exponetial is $1/\lambda$, so this means that the average days to adoption is

In [ ]:
plot(density(1/as.vector(trace_a$draws("Adoption_rate"))), col = "dodgerblue", lwd = 2,
     main = "", xlab = 'Average days to adoption')
abline(v = mean(D))
# savefig('meandays.jpg')

But in our dataset we stopped recording the data at some point, with newly-arrived cats that hadn't yet had a chance to be adopted. So how do we deal with this right-hand censoring? Well for the cats that are adopted we know their number of days to adoption, so no issues there. But for the cats in the data that have yet to adopted, all we know is how long they've been in jail. This second set of cats implies a different quantity, namely the number of days in captivity without yet having been adpoted, which is complement of the exponential (i.e. the complementary cumulative distribution). We can add this into our model as a special component:

$$
\begin{align}
f(c=1,D) = & \lambda e^{-\lambda D} \\
f(c=0,D) = & e^{-\lambda D}
\end{align}
$$

The top line is just the Exponential PDF. The bottom line is the censored PDF, which simply has the leading $\lambda$ parameter removed. This because the uncensorted cumulative distribution function is

$$
1-e^{-\lambda D}
$$

which is the probability of being adopted given time D. So to get the censored bit, this is simply the probability of *not* being adopted by time D, which is

$$
e^{-\lambda D}.
$$

Because of this small difference, we can get the log probability for both kinds of data (censored and not) by adding an indicator variable (c) to show that the cat has or has not been adopted


$$
log(\lambda e^{-\lambda D})-log(e^{-\lambda D})c
$$


$$
log(\lambda)c -\lambda D
$$

cool huh?

In [ ]:
# Adoped or not
A <- as.integer(cdata$out_event == 'Adoption')

In Stan we don't have to hand-code that density at all. Writing `target +=` adds whatever we like to the log-probability, and Stan already supplies both pieces: `exponential_lpdf(days | λ)` is the log of the exponential density (the cats that were adopted), while `exponential_lccdf(days | λ)` is the log *complementary* cumulative distribution - the log probability of surviving at least that long without being adopted, which is exactly the $e^{-\lambda D}$ term above. So the two branches of the model below are just the two lines of maths above, written out.

In [ ]:
adoptC_code <- "
data {
  int<lower=0> N;
  vector[N] days;
  array[N] int<lower=0, upper=1> adopted;
}
parameters {
  real<lower=0> Adoption_rate;   // Adoption rate
}
model {
  // Adoption rate
  Adoption_rate ~ exponential(1);

  // Censored/non log-probability
  for (i in 1:N) {
    if (adopted[i] == 1)
      target += exponential_lpdf(days[i] | Adoption_rate);    // adopted
    else
      target += exponential_lccdf(days[i] | Adoption_rate);   // not yet adopted
  }
}
"
adoptC <- cmdstan_model(write_stan_file(adoptC_code))

In [ ]:
trace_aC <- sample_stan(adoptC, list(N = length(D), days = D, adopted = A))

In [ ]:
trace_aC$summary()

In [ ]:
cens <- 1/as.vector(trace_aC$draws("Adoption_rate"))
naive <- 1/as.vector(trace_a$draws("Adoption_rate"))

plot_hists(cbind(cens, naive), labels = c('Censored', 'Naive'),
           xlab = 'Average days to adoption')
# savefig('censor.jpg')

This has made a massive difference, with censoring accounting for a 50% decrease in apparent adoption times in the previous model. 


The next question is, are black cats adopted at a lower rate relative to other colours?

In [ ]:
tmp <- indexall(cdata$color == 'Black')
Ic <- tmp[[2]]
Colour <- c('Other', 'Black')

This is the same censored model, with the single adoption rate replaced by a `vector[K]` of rates indexed by colour:

In [ ]:
adoptCb_code <- "
data {
  int<lower=0> N;
  int<lower=1> K;                        // number of colours
  array[N] int<lower=1, upper=K> Ic;     // colour index
  vector[N] days;
  array[N] int<lower=0, upper=1> adopted;
}
parameters {
  vector<lower=0>[K] Adoption_rate;      // Adoption rate
}
model {
  // Adoption rate
  Adoption_rate ~ exponential(1);

  // Censored/non log-probability
  for (i in 1:N) {
    if (adopted[i] == 1)
      target += exponential_lpdf(days[i] | Adoption_rate[Ic[i]]);    // adopted
    else
      target += exponential_lccdf(days[i] | Adoption_rate[Ic[i]]);   // not yet adopted
  }
}
"
adoptCb <- cmdstan_model(write_stan_file(adoptCb_code))

In [ ]:
trace_aCb <- sample_stan(adoptCb, list(N = length(D), K = length(Colour), Ic = Ic,
                                       days = D, adopted = A))

In [ ]:
rates <- trace_aCb$draws("Adoption_rate", format = "matrix")
colnames(rates) <- paste0('Adoption rate[', Colour, ']')
mcmc_intervals(rates)

In [ ]:
other <- 1/rates[, 1]
black <- 1/rates[, 2]

plot_hists(cbind(other, black), labels = c('Other', 'Black'), xlab = 'Days to adoption',
           cols = c("dodgerblue", "black"))
# savefig('black.jpg')

Clearly yes, with the average difference being

In [ ]:
hist(black - other, freq = FALSE, col = 'purple', border = "white",
     main = "", xlab = 'Black cat extra days')
# savefig('blackother.jpg')

about 9 additional days.

## Drunken monks

Mixture models have some incredible properties and the ability to distinguish between different sources of zeros is one of them. What do we mean here? Well often processes aren't distinct in how data can arise - many times multiple things happen that can lead to the same observation. Zeros are a classic example - in fisheries for example, zeros can arise because fish weren't there, or because a boat failed to catch fish that were. These kinds of situations arise all the time and as a result a series of **zero inflated** models have been developed to address them. 

By way of example, imagine a monestary where monks are busy copying manuscripts (important work for a monk) and each day a large number of monks finish copying a small number of manuscripts. This kind of thing could easliy be represented as a Poisson process, whereby manuscripts are churned out at a constant underlying rate ($\lambda$), leading to a Poisson distribution that has some proportion of zeros.

But this being a Catholic monestary the monks on occasion turn to drink, upon which days they produce nothing - an additional source of zeros that is unrelated to the typical rate of production. How should we handle this? Well we can model both processes and allow the model to sort out which zeros are likely to arise from which processes. How? Well let's simulate this process for a year

In [ ]:
# Number of days
N <- 365

# Probability of drinking
p_drink <- 0.2
# Rate of manuscript production
work_rate <- 1.3

# Simulate work or drink for each day of the year
work <- rbinom(N, 1, 1 - p_drink)

# Simulate production of manuscripts
M_ <- rpois(N, work_rate)
M <- work*M_

In [ ]:
# Manuscripts on work days only
brks <- seq(-0.5, max(M_) + 0.5, by = 1)
hist(M_, breaks = brks, ylim = c(0, 170), col = "dodgerblue", border = "white",
     main = "", xlab = 'Manuscripts')
# savefig('work.jpg')

In [ ]:
# Observed manuscript production
hist(M, breaks = brks, ylim = c(0, 170), col = "dodgerblue", border = "white",
     main = "", xlab = 'Manuscripts')
# savefig('manuscripts.jpg')

In [ ]:
# ZI manuscript production
z_work <- sum(M_ == 0)
z_total <- sum(M == 0)

hist(M, breaks = brks, ylim = c(0, 170), col = "dodgerblue", border = "white",
     main = "", xlab = 'Manuscripts')
lines(c(0.09, 0.09), c(z_work, z_total), lwd = 10, col = 'red')
# savefig('zimonk.jpg')

Ok, with our simulated data in place, we need to develop a likelihood that reflects both processes, a zero-inflated Poisson. There is a ZIP model built into both Stan and PyMC, but it's fairly simple to define so let's do that for our model here as well, just to get a bit of practice and the knowledge that **we're allowed** to build whatever model we like - there are no rules beyond coherence.

From our simulation we have the elements we'll need, a binomial for the workin' vs drinkin' days, and a Poisson for the manuscript production rate. First the bionomial pmf (probability **mass** function) for 1/0 data is a Bernoulli

$$
p^{k}(1-p)^{1-k}
$$

and the pmf for a Poisson is

$$
\frac{\lambda^{k}e^{-\lambda}}{k!}
$$

So the probability of a zero is

$$
p + (1-p)\frac{\lambda^{0}e^{-\lambda}}{0!} = p + (1-p)e^{-\lambda}
$$

And the probabilty of a count (i.e. when $k\neq0$) is

$$
(1-p)\frac{\lambda^{k}e^{-\lambda}}{k!}
$$

the Poisson multiplied by the probability of working. In Stan we add these two pieces together ourselves: `log_sum_exp(a, b)` is the numerically safe way to add two probabilities that are being carried around as logs, so the zero case is the drinking branch plus the working-but-produced-nothing branch, and any count above zero can only have come from the working branch - note that here `pDrink` is the probability of zeros (not of counts):

In [ ]:
Monks_code <- "
data {
  int<lower=0> N;
  array[N] int<lower=0> M;
}
parameters {
  real logMrate;   // Manuscript log-scale rate
  real loDrink;    // log-odds of drinking
}
transformed parameters {
  // link functions
  real Mrate = exp(logMrate);
  real pDrink = inv_logit(loDrink);
}
model {
  // Manuscript log-scale rate
  logMrate ~ normal(2, 0.5);

  // log-odds of drinking
  loDrink ~ normal(-1.5, 1);

  // Likelihood
  for (n in 1:N) {
    if (M[n] == 0)
      target += log_sum_exp(bernoulli_lpmf(1 | pDrink),
                            bernoulli_lpmf(0 | pDrink) + poisson_lpmf(0 | Mrate));
    else
      target += bernoulli_lpmf(0 | pDrink) + poisson_lpmf(M[n] | Mrate);
  }
}
generated quantities {
  // Pointwise log-likelihood, for WAIC
  vector[N] log_lik;
  for (n in 1:N) {
    if (M[n] == 0)
      log_lik[n] = log_sum_exp(bernoulli_lpmf(1 | pDrink),
                               bernoulli_lpmf(0 | pDrink) + poisson_lpmf(0 | Mrate));
    else
      log_lik[n] = bernoulli_lpmf(0 | pDrink) + poisson_lpmf(M[n] | Mrate);
  }
}
"
Monks <- cmdstan_model(write_stan_file(Monks_code))

In [ ]:
trace_m <- sample_stan(Monks, list(N = N, M = M))

In [ ]:
tmp <- trace_m$summary(c("logMrate", "loDrink", "Mrate", "pDrink"))
tmp

In [ ]:
hist(as.vector(trace_m$draws("pDrink")), freq = FALSE, col = "dodgerblue", border = "white",
     main = "", xlab = 'P(drinking)')
abline(v = p_drink, col = 'red', lwd = 4)
# savefig('Pdrinking.jpg')

So we were able to recapture the simulated data and see that monks drink about 20% of the time. A question is - what would we have estimated if we'd just used the Poisson?

In [ ]:
PMonks_code <- "
data {
  int<lower=0> N;
  array[N] int<lower=0> M;
}
parameters {
  real logMrate;   // Manuscript log-scale rate
}
transformed parameters {
  // link function
  real Mrate = exp(logMrate);
}
model {
  // Manuscript log-scale rate
  logMrate ~ normal(1, 0.5);

  // Likelihood
  M ~ poisson(Mrate);
}
generated quantities {
  // Pointwise log-likelihood, for WAIC
  vector[N] log_lik;
  for (n in 1:N) log_lik[n] = poisson_lpmf(M[n] | Mrate);
}
"
PMonks <- cmdstan_model(write_stan_file(PMonks_code))

In [ ]:
trace_pm <- sample_stan(PMonks, list(N = N, M = M))

In [ ]:
trace_pm$summary(c("logMrate", "Mrate"))

In [ ]:
Mrates <- cbind(ZIP = as.vector(trace_m$draws("Mrate")),
                Poisson = as.vector(trace_pm$draws("Mrate")))
mcmc_intervals(Mrates)
# ggsave('monksmodels.jpg', dpi = 300)

As you might expect, all those drinking zeros get sunk into the Poisson density and what we estimate is the Monk's overall (both drinking and working days) rate of manuscript production. What would WAIC say comparing these two models?

`loo_compare()` is the R equivalent of `pm.compare()`: it puts the best-supported model at the top and reports how far each other model falls behind it in expected log predictive density (`elpd_diff`, so higher is better and the differences are negative). It doesn't report Akaike-style model weights, so read the gap relative to its standard error instead.

In [ ]:
# WAIC-based model comparison
comp_WAIC <- loo_compare(list(Poisson = waic(trace_pm$draws("log_lik", format = "matrix")),
                              ZIP = waic(trace_m$draws("log_lik", format = "matrix"))))
comp_WAIC

Most of the model weight favours zero-inflation. Note that depending on your simulation above, you may randomly get a dataset that doesn't conform quite as well. Re-run the simulation and see what happens. None of this stuff is easy...

# Multinomial and categorical models

While binomials are commonplace for looking at 0/1 data, successes in trials, categorical data can apply to mutliple outcomes, where quantities end up on distinct bins. 

Let's look at this kind of data through a simulated example, whereby we simulate various career choices based on expected income. We can port over McElreath's code from Chatper 11 to do so:

In [ ]:
# Simulate career choices for 500 people
N <- 500
# Expected income for each career
income <- c(1, 2, 5)
# Score for each career based on income
bx <- 0.4
score <- 0.4*income

# Convert scores to probabilities
p <- softmax(score)

In [ ]:
p

In [ ]:
# Simulate choices based on underlying income
career <- rmultinom(N, 1, p)
# Put choice into categorical array
career <- apply(career, 2, which.max)
career

So with these in place, we can develop a regression model to figure out what the effect of expected income is on career choice:

In [ ]:
Careerz_code <- "
data {
  int<lower=0> N;
  int<lower=2> K;                            // number of careers
  vector[K] income;
  array[N] int<lower=1, upper=K> career;
}
parameters {
  vector[K-1] a;         // Intercepts
  real<lower=0> b;       // Income slope
}
transformed parameters {
  // Linear model
  vector[K] s;
  s[1] = a[1] + b*income[1];
  s[2] = a[2] + b*income[2];
  s[3] = 0;
}
model {
  a ~ normal(0, 1);
  b ~ lognormal(0, 0.5);

  // Data likelihood
  career ~ categorical_logit(s);
}
"
Careerz <- cmdstan_model(write_stan_file(Careerz_code))

In [ ]:
trace_c <- sample_stan(Careerz, list(N = N, K = length(income), income = income, career = career))

In [ ]:
tmp <- trace_c$summary(c("a", "b", "s"))
tmp

Great - we have results, but unfortunately the nature of multinomial regression is that effect sizes are really difficult to interpret, because they're relative to the baseline category used (the 'pivot'). So instead let's look at the posterior predicted probabilities for each category, relative to our known values:

In [ ]:
trace_c$draws("s")

In [ ]:
# Estimated probabilities
s_draws <- trace_c$draws("s", format = "matrix")
smax <- t(apply(s_draws, 1, softmax))
softmax(colMeans(s_draws))

In [ ]:
plot_hists(smax, labels = as.character(1:3), xlab = 'P(career)', vline = p)
# savefig('Pcareer.jpg')

These look great - we've captured the probabilities of choosing each career. But that's a bit unsatisfying, as what we really want to know is the relationship between incomes and choosing each careear. To gain some insight to this, the thing to do is simulate what would happen to these probabilites if, say, the income for career=1 doubles:

In [ ]:
# Grab posteriors
a_post <- trace_c$draws("a", format = "matrix")
s_post <- trace_c$draws("s", format = "matrix")
b_post <- as.vector(trace_c$draws("b"))

In [ ]:
hist(b_post, col = "dodgerblue", border = "white", main = "", xlab = 'Income effect on career')
abline(v = bx, col = 'red', lwd = 4)
# savefig('Pcareerb.jpg')

In [ ]:
# Set up logit scores
s0x <- a_post[, 1] + b_post*income[1]
s1x <- a_post[, 2] + b_post*income[2]
s1x_new <- a_post[, 2] + b_post*income[2]*2

In [ ]:
# Calculate probabilites from original income and new income
p_orig <- t(apply(cbind(s0x, s1x, 0), 1, softmax))
p_new <- t(apply(cbind(s0x, s1x_new, 0), 1, softmax))

So plotting our model posteriors against our 

In [ ]:
plot_hists(p_orig, labels = as.character(1:3), xlab = 'P(career)')
# savefig('Pcareer1.jpg')

In [ ]:
plot_hists(p_new, labels = as.character(1:3), xlab = 'P(career)')
# savefig('Pcareer2.jpg')